<a href="https://colab.research.google.com/github/SUNIL-KUMAR0869/Doc-Genie-Python-Analyzer/blob/main/DOC_GENIE_ANALYZER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import ast
import gradio as gr
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import os

class DocGenieAnalyzer:
    def analyze_and_generate(self, code_string):
        try:
            tree = ast.parse(code_string)
            functions = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
            if not functions: return "Error: No function found.", None

            func = functions[0]
            args = [arg.arg for arg in func.args.args]

            # Logic Analysis
            logic = []
            for node in ast.walk(func):
                if isinstance(node, (ast.For, ast.While)): logic.append("Loops/Iteration")
                if isinstance(node, ast.If): logic.append("Conditional Logic")

            # Generate Docstring
            doc = f'"""{func.name.replace("_", " ").capitalize()}.\n\nArgs:\n'
            for a in args: doc = doc + f"    {a}: Input parameter.\n"
            if logic: doc = doc + f"\nDetected: {', '.join(set(logic))}\n"
            doc += '"""'

            # Create PDF
            pdf_path = "documentation.pdf"
            c = canvas.Canvas(pdf_path, pagesize=letter)
            c.drawString(100, 750, f"Doc-Genie Report: {func.name}")
            textobject = c.beginText(100, 730)
            for line in doc.split('\n'):
                textobject.textLine(line)
            c.drawText(textobject)
            c.save()

            return doc, pdf_path
        except Exception as e:
            return f"Error: {str(e)}", None

genie = DocGenieAnalyzer()

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# Doc-Genie: AI Documentation Assistant")
    code_input = gr.Textbox(lines=10, label="Input Python Code")
    doc_output = gr.Code(label="Generated Docstring", language="python")
    file_output = gr.File(label="Download PDF Report")
    submit_btn = gr.Button("Generate & Export")

    submit_btn.click(fn=genie.analyze_and_generate, inputs=code_input, outputs=[doc_output, file_output])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6dcfe2366129e84d2f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
